# 02 | Reporting Model and Exports

This notebook **does not perform a new analysis**. It reproduces the accepted logic from `01_market_questions.ipynb` using `data/processed/` and exports the minimum set of tables required in `data/reporting/`.

## Outputs

1. `dim_city.csv`
2. `dim_year.csv`
3. `dim_nbp_period.csv`
4. `fact_price_period.csv`
5. `fact_city_year.csv`
6. `fact_hedonic_period.csv`
7. `city_snapshot.csv`

Model rule: **dimension → fact**, with no fact to fact relationships.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 0. Paths and processed data import

The notebook locates the project root by searching for `data/processed/`, so it can be executed from the `notebooks/` directory.

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError(
        "data/processed was not found. Run the notebook inside the project repository."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTING_DIR = PROJECT_ROOT / "data" / "reporting"
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "dim_city": PROCESSED_DIR / "dim_city.csv",
    "prices": PROCESSED_DIR / "nbp_prices_clean.csv",
    "hedonic": PROCESSED_DIR / "nbp_hedonic_clean.csv",
    "income": PROCESSED_DIR / "gus_income_clean.csv",
    "market": PROCESSED_DIR / "gus_market_clean.csv",
    "supply": PROCESSED_DIR / "gus_supply_clean.csv",
    "population": PROCESSED_DIR / "gus_population_clean.csv",
}

missing_files = [path.name for path in FILES.values() if not path.exists()]
assert not missing_files, f"Missing processed files: {missing_files}"

dim_city_raw = pd.read_csv(FILES["dim_city"], dtype={"teryt_code": "string"})
prices = pd.read_csv(FILES["prices"], dtype={"teryt_code": "string"})
hedonic = pd.read_csv(FILES["hedonic"])
income = pd.read_csv(FILES["income"], dtype={"teryt_code": "string"})
market = pd.read_csv(FILES["market"], dtype={"teryt_code": "string"})
supply = pd.read_csv(FILES["supply"], dtype={"teryt_code": "string"})
population = pd.read_csv(FILES["population"], dtype={"teryt_code": "string"})

assert dim_city_raw["teryt_code"].str.fullmatch(r"\d{7}").all()

print("Processed inputs: data/processed")
print("Reporting outputs: data/reporting")

Processed inputs: data/processed
Reporting outputs: data/reporting


In [3]:
expected_rows = {
    "dim_city": 17,
    "prices": 5_440,
    "hedonic": 1_280,
    "income": 408,
    "market": 1_020,
    "supply": 8_772,
    "population": 1_054,
}

loaded = {
    "dim_city": dim_city_raw,
    "prices": prices,
    "hedonic": hedonic,
    "income": income,
    "market": market,
    "supply": supply,
    "population": population,
}

input_check = pd.DataFrame({
    "rows": {name: len(df) for name, df in loaded.items()},
    "expected_rows": expected_rows,
})
input_check["ok"] = input_check["rows"] == input_check["expected_rows"]

display(input_check)
assert input_check["ok"].all(), "Processed input does not match the accepted handoff."

,rows,expected_rows,ok
dim_city,17,17,True
prices,5440,5440,True
hedonic,1280,1280,True
income,408,408,True
market,1020,1020,True
supply,8772,8772,True
population,1054,1054,True


# 1. Dimensions

* `dim_city[city_id]` filters all tables.
* `dim_year[year]` filters `fact_city_year`.
* `dim_nbp_period[nbp_period_id]` filters quarterly NBP fact tables.

City names remain in the dimension while fact tables store the `city_id` key.

In [4]:
dim_city = (
    dim_city_raw[["city_id", "city_name", "teryt_code", "gus_name"]]
    .drop_duplicates()
    .sort_values("city_id")
    .reset_index(drop=True)
)

assert len(dim_city) == 17
assert dim_city["city_id"].is_unique
assert dim_city["city_name"].is_unique

dim_nbp_period = (
    prices[["nbp_period_order", "nbp_period", "nbp_year", "nbp_quarter"]]
    .drop_duplicates()
    .rename(columns={"nbp_period_order": "nbp_period_id"})
    .sort_values("nbp_period_id")
    .reset_index(drop=True)
)

assert len(dim_nbp_period) == 80
assert dim_nbp_period["nbp_period_id"].is_unique
assert dim_nbp_period["nbp_period"].is_unique

dim_year = pd.DataFrame({"year": range(2005, 2027)})

display(dim_city.head())
display(dim_nbp_period.tail())
display(dim_year.tail())

,city_id,city_name,teryt_code,gus_name
0,1,Białystok,2061000,Powiat m. Białystok
1,2,Bydgoszcz,0461000,Powiat m. Bydgoszcz
2,3,Gdańsk,2261000,Powiat m. Gdańsk
3,4,Gdynia,2262000,Powiat m. Gdynia
4,5,Katowice,2469000,Powiat m. Katowice


,nbp_period_id,nbp_period,nbp_year,nbp_quarter
75,20252,2025Q2,2025,2
76,20253,2025Q3,2025,3
77,20254,2025Q4,2025,4
78,20261,2026Q1,2026,1
79,20262,2026Q2,2026,2


,year
17,2022
18,2023
19,2024
20,2025
21,2026


# 2. `fact_price_period`

**Grain:** `city_id × nbp_period_id`

This table mainly supports questions about the latest price, primary versus secondary pricing, and price momentum.

Momentum includes:

* year over year transaction price growth,
* average year over year growth across the latest 4 periods,
* the difference versus the previous 4 periods.

A negative `*_momentum_change_pp` means that the pace of change is weaker than one year earlier. (It is not a forecast)

In [5]:
transaction_prices = prices.loc[
    prices["price_type"].eq("transaction"),
    [
        "city_id",
        "nbp_period_order",
        "market_type",
        "price_pln_m2",
    ],
].copy()

price_base = (
    transaction_prices
    .pivot(
        index=["city_id", "nbp_period_order"],
        columns="market_type",
        values="price_pln_m2",
    )
    .rename(columns={
        "primary": "primary_transaction_price_pln_m2",
        "secondary": "secondary_transaction_price_pln_m2",
    })
    .reset_index()
    .rename(columns={"nbp_period_order": "nbp_period_id"})
    .sort_values(["city_id", "nbp_period_id"])
    .reset_index(drop=True)
)

for market_type in ["primary", "secondary"]:
    price_col = f"{market_type}_transaction_price_pln_m2"
    yoy_col = f"{market_type}_transaction_yoy_pct"
    rolling_col = f"{market_type}_yoy_4p_avg_pct"
    momentum_col = f"{market_type}_momentum_change_pp"

    price_base[yoy_col] = (
        price_base.groupby("city_id")[price_col]
        .pct_change(periods=4, fill_method=None)
        * 100
    )

    price_base[rolling_col] = (
        price_base.groupby("city_id")[yoy_col]
        .transform(lambda s: s.rolling(4, min_periods=4).mean())
    )

    price_base[momentum_col] = (
        price_base[rolling_col]
        - price_base.groupby("city_id")[rolling_col].shift(4)
    )

price_base["primary_secondary_premium_pct"] = (
    price_base["primary_transaction_price_pln_m2"]
    / price_base["secondary_transaction_price_pln_m2"]
    - 1
) * 100

## Gdynia methodology breakpoint

The reporting layer only needs the **primary transaction** breakpoint at `2021Q3` because offer prices are not exported to Power BI.

The observation is retained. A flag and note are exported so the breakpoint can be shown in a tooltip or visual annotation.

In [18]:
methodology_transaction = (
    prices.loc[
        prices["market_type"].eq("primary")
        & prices["price_type"].eq("transaction")
        & prices["methodology_break"].eq(True),
        [
            "city_id",
            "nbp_period_order",
            "methodology_break",
            "methodology_note",
        ],
    ]
    .rename(columns={
        "nbp_period_order": "nbp_period_id",
        "methodology_break": "primary_transaction_methodology_break",
        "methodology_note": "primary_transaction_methodology_note",
    })
    .copy()
)

fact_price_period = price_base.merge(
    methodology_transaction,
    on=["city_id", "nbp_period_id"],
    how="left",
    validate="one_to_one",
)

fact_price_period["primary_transaction_methodology_break"] = (
    fact_price_period["primary_transaction_methodology_break"]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

assert len(fact_price_period) == 17 * 80
assert not fact_price_period.duplicated(["city_id", "nbp_period_id"]).any()

display(fact_price_period.tail())

,city_id,nbp_period_id,primary_transaction_price_pln_m2,secondary_transaction_price_pln_m2,primary_transaction_yoy_pct,primary_yoy_4p_avg_pct,primary_momentum_change_pp,secondary_transaction_yoy_pct,secondary_yoy_4p_avg_pct,secondary_momentum_change_pp,primary_secondary_premium_pct,primary_transaction_methodology_break,primary_transaction_methodology_note
1355,17,20252,"10,051.23","8,154.39",-2.61,5.07,-5.83,4.48,10.07,-1.65,23.26,False,NaN
1356,17,20253,"9,837.47","8,243.16",-1.16,1.28,-12.36,2.85,6.50,-9.14,19.34,False,NaN
1357,17,20254,"9,707.98","8,005.39",-2.88,-1.60,-15.40,2.17,4.00,-12.67,21.27,False,NaN
1358,17,20261,"9,757.71","8,008.83",-0.24,-1.72,-11.73,-0.49,2.25,-12.35,21.84,False,NaN
1359,17,20262,"9,945.49","8,019.66",-1.05,-1.33,-6.41,-1.65,0.72,-9.35,24.01,False,NaN


# 3. `fact_city_year`

**Grain:** `city_id × year`

This is the main annual cross source table.

The source ranges remain different:

* affordability and annual prices: through 2025,
* market activity: through 2024,
* full year supply: through 2025,
* H1 supply: through 2026.

Missing data remains missing.

## 3.1 Population and income

In [7]:
population_mid = (
    population.loc[
        population["reference_date"].eq("mid_year"),
        ["city_id", "year", "population"],
    ]
    .rename(columns={"population": "population_mid_year"})
    .copy()
)

income_year = income[
    ["city_id", "year", "avg_gross_monthly_wage_pln"]
].copy()

assert not population_mid.duplicated(["city_id", "year"]).any()
assert not income_year.duplicated(["city_id", "year"]).any()

## 3.2 Annual transaction prices and affordability

An annual price is the mean of 4 NBP observations within a year. An incomplete year remains `NA`.

`affordability = average gross monthly wage / annual transaction price per m²`

In [8]:
annual_price_long = (
    prices.loc[
        prices["price_type"].eq("transaction")
        & prices["nbp_year"].between(2007, 2025)
    ]
    .groupby(["city_id", "nbp_year", "market_type"], as_index=False)
    .agg(
        annual_transaction_price_pln_m2=("price_pln_m2", "mean"),
        quarters_available=("price_pln_m2", "count"),
    )
    .rename(columns={"nbp_year": "year"})
)

annual_price_long.loc[
    annual_price_long["quarters_available"].lt(4),
    "annual_transaction_price_pln_m2",
] = np.nan

annual_price_wide = (
    annual_price_long
    .pivot(
        index=["city_id", "year"],
        columns="market_type",
        values=["annual_transaction_price_pln_m2", "quarters_available"],
    )
)

annual_price_wide.columns = [
    f"{market}_{metric}"
    for metric, market in annual_price_wide.columns
]

annual_price_wide = (
    annual_price_wide
    .reset_index()
    .rename(columns={
        "primary_quarters_available": "primary_price_quarters_available",
        "secondary_quarters_available": "secondary_price_quarters_available",
    })
)

annual_affordability = (
    annual_price_wide
    .merge(
        income_year,
        on=["city_id", "year"],
        how="left",
        validate="many_to_one",
    )
    .sort_values(["city_id", "year"])
)

for market_type in ["primary", "secondary"]:
    price_col = f"{market_type}_annual_transaction_price_pln_m2"
    aff_col = f"{market_type}_affordability_m2"

    annual_affordability[aff_col] = (
        annual_affordability["avg_gross_monthly_wage_pln"]
        / annual_affordability[price_col]
    )

    annual_affordability[f"{market_type}_annual_price_yoy_pct"] = (
        annual_affordability.groupby("city_id")[price_col]
        .pct_change(fill_method=None)
        * 100
    )

    annual_affordability[f"{market_type}_affordability_yoy_pct"] = (
        annual_affordability.groupby("city_id")[aff_col]
        .pct_change(fill_method=None)
        * 100
    )

annual_affordability["wage_yoy_pct"] = (
    annual_affordability.groupby("city_id")["avg_gross_monthly_wage_pln"]
    .pct_change(fill_method=None)
    * 100
)

## 3.3 Annual market activity

In [9]:
market_annual = (
    market
    .groupby(["city_id", "year"], as_index=False)
    .agg(
        transactions=("transactions", lambda s: s.sum(min_count=4)),
        transaction_quarters_available=("transactions", "count"),
        sold_units=("sold_units", lambda s: s.sum(min_count=4)),
        sold_units_quarters_available=("sold_units", "count"),
        sold_area_m2=("sold_area_m2", lambda s: s.sum(min_count=4)),
        sold_area_quarters_available=("sold_area_m2", "count"),
    )
)

market_annual["avg_sold_area_m2"] = (
    market_annual["sold_area_m2"] / market_annual["sold_units"]
)

market_annual["transactions_full_year_available"] = (
    market_annual["transaction_quarters_available"].eq(4)
)
market_annual["sold_units_full_year_available"] = (
    market_annual["sold_units_quarters_available"].eq(4)
)

market_annual = market_annual.merge(
    population_mid,
    on=["city_id", "year"],
    how="left",
    validate="one_to_one",
)

market_annual["transactions_per_1000"] = (
    market_annual["transactions"]
    / market_annual["population_mid_year"]
    * 1_000
)

market_annual["sold_units_per_1000"] = (
    market_annual["sold_units"]
    / market_annual["population_mid_year"]
    * 1_000
)

## 3.4 Supply: full year and H1

A full year requires 12 months of `period_value`.

H1 is calculated separately so that 2026 is compared with H1 2025 rather than with full year 2025.

In [10]:
supply_full_long = (
    supply.loc[supply["is_full_year"].eq(True)]
    .groupby(["city_id", "year", "supply_type"], as_index=False)
    .agg(
        dwellings=("period_value", lambda s: s.sum(min_count=12)),
        months_available=("period_value", "count"),
    )
)

supply_full_wide = (
    supply_full_long
    .pivot(
        index=["city_id", "year"],
        columns="supply_type",
        values=["dwellings", "months_available"],
    )
)

supply_full_wide.columns = [
    f"{supply_type}_{metric}"
    for metric, supply_type in supply_full_wide.columns
]

supply_full_wide = (
    supply_full_wide
    .reset_index()
    .rename(columns={
        "started_dwellings": "started",
        "completed_dwellings": "completed",
        "started_months_available": "started_months_available",
        "completed_months_available": "completed_months_available",
    })
)

supply_full_wide["started_full_year_available"] = (
    supply_full_wide["started_months_available"].eq(12)
)
supply_full_wide["completed_full_year_available"] = (
    supply_full_wide["completed_months_available"].eq(12)
)

supply_full_wide = supply_full_wide.merge(
    population_mid,
    on=["city_id", "year"],
    how="left",
    validate="one_to_one",
)

supply_full_wide["started_per_1000"] = (
    supply_full_wide["started"]
    / supply_full_wide["population_mid_year"]
    * 1_000
)
supply_full_wide["completed_per_1000"] = (
    supply_full_wide["completed"]
    / supply_full_wide["population_mid_year"]
    * 1_000
)

supply_h1_long = (
    supply.loc[supply["month"].le(6)]
    .groupby(["city_id", "year", "supply_type"], as_index=False)
    .agg(
        h1_dwellings=("period_value", lambda s: s.sum(min_count=6)),
        h1_months_available=("period_value", "count"),
    )
)

supply_h1_wide = (
    supply_h1_long
    .pivot(
        index=["city_id", "year"],
        columns="supply_type",
        values=["h1_dwellings", "h1_months_available"],
    )
)

supply_h1_wide.columns = [
    f"{supply_type}_{metric}"
    for metric, supply_type in supply_h1_wide.columns
]

supply_h1_wide = (
    supply_h1_wide
    .reset_index()
    .rename(columns={
        "started_h1_dwellings": "started_h1",
        "completed_h1_dwellings": "completed_h1",
        "started_h1_months_available": "started_h1_months_available",
        "completed_h1_months_available": "completed_h1_months_available",
    })
    .sort_values(["city_id", "year"])
)

for supply_type in ["started", "completed"]:
    supply_h1_wide[f"{supply_type}_h1_yoy_pct"] = (
        supply_h1_wide.groupby("city_id")[f"{supply_type}_h1"]
        .pct_change(fill_method=None)
        * 100
    )

## 3.5 Assemble the annual table

In [11]:
city_year_grid = (
    pd.MultiIndex.from_product(
        [dim_city["city_id"], dim_year["year"]],
        names=["city_id", "year"],
    )
    .to_frame(index=False)
)

fact_city_year = (
    city_year_grid
    .merge(
        income_year,
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        population_mid,
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        annual_affordability.drop(
            columns=["avg_gross_monthly_wage_pln"],
            errors="ignore",
        ),
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        market_annual.drop(
            columns=["population_mid_year"],
            errors="ignore",
        ),
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        supply_full_wide.drop(
            columns=["population_mid_year"],
            errors="ignore",
        ),
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        supply_h1_wide,
        on=["city_id", "year"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["city_id", "year"])
    .reset_index(drop=True)
)

assert len(fact_city_year) == 17 * len(dim_year)
assert not fact_city_year.duplicated(["city_id", "year"]).any()

display(fact_city_year.tail())

,city_id,year,avg_gross_monthly_wage_pln,population_mid_year,primary_annual_transaction_price_pln_m2,secondary_annual_transaction_price_pln_m2,primary_price_quarters_available,secondary_price_quarters_available,primary_affordability_m2,primary_annual_price_yoy_pct,primary_affordability_yoy_pct,secondary_affordability_m2,secondary_annual_price_yoy_pct,secondary_affordability_yoy_pct,wage_yoy_pct,transactions,transaction_quarters_available,sold_units,sold_units_quarters_available,sold_area_m2,sold_area_quarters_available,avg_sold_area_m2,transactions_full_year_available,sold_units_full_year_available,transactions_per_1000,sold_units_per_1000,completed,started,completed_months_available,started_months_available,started_full_year_available,completed_full_year_available,started_per_1000,completed_per_1000,completed_h1,started_h1,completed_h1_months_available,started_h1_months_available,started_h1_yoy_pct,completed_h1_yoy_pct
369,17,2022,"6,629.23","661,329.00","8,160.60","6,513.22",4.00,4.00,0.81,15.94,-5.68,1.02,10.79,-1.30,9.36,"6,986.00",4.00,"7,168.00",4.00,"348,920.10",4.00,48.68,True,True,10.56,10.84,"5,788.00","4,003.00",12.00,12.00,True,True,6.05,8.75,"3,199.00","1,883.00",6.00,6.00,-51.94,33.51
370,17,2023,"7,548.10","655,279.00","8,798.33","6,694.43",4.00,4.00,0.86,7.81,5.61,1.13,2.78,10.78,13.86,"8,858.00",4.00,"8,629.00",4.00,"417,746.80",4.00,48.41,True,True,13.52,13.17,"5,204.00","5,037.00",12.00,12.00,True,True,7.69,7.94,"2,073.00","2,452.00",6.00,6.00,30.22,-35.20
371,17,2024,"8,621.77","648,711.00","10,006.26","7,803.06",4.00,4.00,0.86,13.73,0.44,1.10,16.56,-2.00,14.22,"9,679.00",4.00,"9,633.00",4.00,"468,817.80",4.00,48.67,True,True,14.92,14.85,"5,489.00","10,413.00",12.00,12.00,True,True,16.05,8.46,"2,522.00","6,999.00",6.00,6.00,185.44,21.66
372,17,2025,"9,374.53","642,590.00","9,844.45","8,112.78",4.00,4.00,0.95,-1.62,10.52,1.16,3.97,4.58,8.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"5,792.00","7,520.00",12.00,12.00,True,True,11.70,9.01,"2,263.00","4,366.00",6.00,6.00,-37.62,-10.27
373,17,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,602.00","3,182.00",6.00,6.00,-27.12,14.98


# 4. `fact_hedonic_period`

**Grain:** `city_id × nbp_period_id`

This table compares secondary market headline average price growth with hedonic year over year growth.

Gdańsk and Gdynia are excluded from the city level comparison because NBP provides the `Trójmiasto` aggregate here rather than two separate city series.

In [12]:
hedonic_city = hedonic.loc[
    hedonic["geography_type"].eq("city")
].copy()

hedonic_city["hedonic_qoq_pct"] = hedonic_city["hedonic_qoq_index"] - 100
hedonic_city["hedonic_yoy_pct"] = hedonic_city["hedonic_yoy_index"] - 100

hedonic_city = hedonic_city.merge(
    dim_city[["city_id", "city_name"]],
    left_on="geography",
    right_on="city_name",
    how="inner",
    validate="many_to_one",
)

hedonic_city = hedonic_city.merge(
    dim_nbp_period[["nbp_period_id", "nbp_period"]],
    on="nbp_period",
    how="left",
    validate="many_to_one",
)

fact_hedonic_period = (
    hedonic_city[
        [
            "city_id",
            "nbp_period_id",
            "hedonic_qoq_index",
            "hedonic_yoy_index",
            "hedonic_qoq_pct",
            "hedonic_yoy_pct",
        ]
    ]
    .merge(
        fact_price_period[
            ["city_id", "nbp_period_id", "secondary_transaction_yoy_pct"]
        ],
        on=["city_id", "nbp_period_id"],
        how="left",
        validate="one_to_one",
    )
)

fact_hedonic_period["headline_minus_hedonic_yoy_pp"] = (
    fact_hedonic_period["secondary_transaction_yoy_pct"]
    - fact_hedonic_period["hedonic_yoy_pct"]
)

comparable = (
    fact_hedonic_period["secondary_transaction_yoy_pct"].notna()
    & fact_hedonic_period["hedonic_yoy_pct"].notna()
)

nonzero = (
    fact_hedonic_period["secondary_transaction_yoy_pct"].ne(0)
    & fact_hedonic_period["hedonic_yoy_pct"].ne(0)
)

fact_hedonic_period["comparison_available"] = comparable
fact_hedonic_period["opposite_direction"] = (
    comparable
    & nonzero
    & (
        np.sign(fact_hedonic_period["secondary_transaction_yoy_pct"])
        != np.sign(fact_hedonic_period["hedonic_yoy_pct"])
    )
)

assert len(fact_hedonic_period) == 15 * 80
assert not fact_hedonic_period.duplicated(
    ["city_id", "nbp_period_id"]
).any()

display(fact_hedonic_period.tail())

,city_id,nbp_period_id,hedonic_qoq_index,hedonic_yoy_index,hedonic_qoq_pct,hedonic_yoy_pct,secondary_transaction_yoy_pct,headline_minus_hedonic_yoy_pp,comparison_available,opposite_direction
1195,13,20254,100.99,101.70,0.99,1.70,1.51,-0.19,True,False
1196,14,20254,103.31,101.76,3.31,1.76,-1.26,-3.02,True,True
1197,15,20254,97.72,97.56,-2.28,-2.44,-2.85,-0.41,True,False
1198,16,20254,104.54,115.74,4.54,15.74,8.04,-7.70,True,False
1199,17,20254,103.45,107.06,3.45,7.06,2.17,-4.89,True,False


# 5. `city_snapshot`

**Grain:** `city_id`

This table provides the latest cross section of cities. Each metric group keeps its own reference period:

* prices: latest NBP period,
* affordability: 2025,
* market activity: 2024,
* full year supply: 2025,
* H1 supply: H1 2026 versus H1 2025.

The model does not pretend that every source has the same level of recency.

In [13]:
latest_price_period_id = int(dim_nbp_period["nbp_period_id"].max())
latest_price_period = dim_nbp_period.loc[
    dim_nbp_period["nbp_period_id"].eq(latest_price_period_id),
    "nbp_period",
].iloc[0]

affordability_year = 2025
market_activity_year = 2024
supply_year = 2025
supply_h1_year = 2026

price_snapshot = fact_price_period.loc[
    fact_price_period["nbp_period_id"].eq(latest_price_period_id)
].copy()

premium_history = (
    fact_price_period
    .groupby("city_id", as_index=False)
    .agg(
        primary_premium_historical_median_pct=(
            "primary_secondary_premium_pct", "median"
        ),
        primary_premium_historical_min_pct=(
            "primary_secondary_premium_pct", "min"
        ),
        primary_premium_historical_max_pct=(
            "primary_secondary_premium_pct", "max"
        ),
    )
)

aff_2025 = fact_city_year.loc[
    fact_city_year["year"].eq(affordability_year),
    [
        "city_id",
        "avg_gross_monthly_wage_pln",
        "primary_annual_transaction_price_pln_m2",
        "secondary_annual_transaction_price_pln_m2",
        "primary_affordability_m2",
        "secondary_affordability_m2",
        "primary_affordability_yoy_pct",
        "secondary_affordability_yoy_pct",
        "primary_annual_price_yoy_pct",
        "secondary_annual_price_yoy_pct",
        "wage_yoy_pct",
    ],
].copy()

aff_2019 = fact_city_year.loc[
    fact_city_year["year"].eq(2019),
    ["city_id", "primary_affordability_m2", "secondary_affordability_m2"],
].rename(columns={
    "primary_affordability_m2": "primary_affordability_2019_m2",
    "secondary_affordability_m2": "secondary_affordability_2019_m2",
})

activity_2024 = fact_city_year.loc[
    fact_city_year["year"].eq(market_activity_year),
    [
        "city_id",
        "sold_units",
        "sold_units_per_1000",
        "transactions",
        "transactions_per_1000",
        "avg_sold_area_m2",
    ],
].copy()

supply_2025 = fact_city_year.loc[
    fact_city_year["year"].eq(supply_year),
    [
        "city_id",
        "started",
        "started_per_1000",
        "completed",
        "completed_per_1000",
    ],
].copy()

supply_h1_2026 = fact_city_year.loc[
    fact_city_year["year"].eq(supply_h1_year),
    [
        "city_id",
        "started_h1",
        "completed_h1",
        "started_h1_yoy_pct",
        "completed_h1_yoy_pct",
    ],
].copy()

city_snapshot = (
    dim_city[["city_id"]]
    .merge(price_snapshot, on="city_id", how="left", validate="one_to_one")
    .merge(premium_history, on="city_id", how="left", validate="one_to_one")
    .merge(aff_2025, on="city_id", how="left", validate="one_to_one")
    .merge(aff_2019, on="city_id", how="left", validate="one_to_one")
    .merge(activity_2024, on="city_id", how="left", validate="one_to_one")
    .merge(supply_2025, on="city_id", how="left", validate="one_to_one")
    .merge(supply_h1_2026, on="city_id", how="left", validate="one_to_one")
)

city_snapshot["primary_premium_vs_historical_median_pp"] = (
    city_snapshot["primary_secondary_premium_pct"]
    - city_snapshot["primary_premium_historical_median_pct"]
)

for market_type in ["primary", "secondary"]:
    city_snapshot[f"{market_type}_affordability_change_since_2019_pct"] = (
        city_snapshot[f"{market_type}_affordability_m2"]
        / city_snapshot[f"{market_type}_affordability_2019_m2"]
        - 1
    ) * 100

city_snapshot["price_reference_period"] = latest_price_period
city_snapshot["affordability_reference_year"] = affordability_year
city_snapshot["market_activity_reference_year"] = market_activity_year
city_snapshot["supply_reference_year"] = supply_year
city_snapshot["supply_h1_reference_year"] = supply_h1_year

assert len(city_snapshot) == 17
assert city_snapshot["city_id"].is_unique

## Pareto set without an arbitrary score

For the secondary market, the favorable directions are:

* higher affordability,
* higher completed dwellings per 1,000 residents,
* lower average year over year growth across the latest 4 periods.

`pareto_favorable_secondary = True` only means that no other city dominates it across all three dimensions at the same time.

The inputs use different latest valid periods: recent price growth is based on quarterly data through 2026Q2, while affordability and completed supply use 2025. This is therefore not a strict same-date cross-section.

In [14]:
def dominates(candidate, other):
    better_or_equal = (
        candidate["secondary_affordability_m2"]
        >= other["secondary_affordability_m2"]
        and candidate["completed_per_1000"]
        >= other["completed_per_1000"]
        and candidate["secondary_yoy_4p_avg_pct"]
        <= other["secondary_yoy_4p_avg_pct"]
    )

    strictly_better_somewhere = (
        candidate["secondary_affordability_m2"]
        > other["secondary_affordability_m2"]
        or candidate["completed_per_1000"]
        > other["completed_per_1000"]
        or candidate["secondary_yoy_4p_avg_pct"]
        < other["secondary_yoy_4p_avg_pct"]
    )

    return better_or_equal and strictly_better_somewhere

pareto_flags = []

for i, row in city_snapshot.iterrows():
    dominated = False

    for j, candidate in city_snapshot.iterrows():
        if i == j:
            continue
        if dominates(candidate, row):
            dominated = True
            break

    pareto_flags.append(not dominated)

city_snapshot["pareto_favorable_secondary"] = pareto_flags

city_snapshot["secondary_price_rank_lowest"] = (
    city_snapshot["secondary_transaction_price_pln_m2"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)
city_snapshot["secondary_affordability_rank_highest"] = (
    city_snapshot["secondary_affordability_m2"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
city_snapshot["completed_supply_rank_highest"] = (
    city_snapshot["completed_per_1000"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
city_snapshot["secondary_price_pressure_rank_lowest"] = (
    city_snapshot["secondary_yoy_4p_avg_pct"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)

display(
    city_snapshot[
        [
            "city_id",
            "secondary_transaction_price_pln_m2",
            "secondary_affordability_m2",
            "completed_per_1000",
            "secondary_yoy_4p_avg_pct",
            "pareto_favorable_secondary",
        ]
    ]
)

,city_id,secondary_transaction_price_pln_m2,secondary_affordability_m2,completed_per_1000,secondary_yoy_4p_avg_pct,pareto_favorable_secondary
0,1,"9,403.13",0.96,5.84,0.66,True
1,2,"8,341.87",1.18,3.65,1.86,False
2,3,"14,034.64",0.82,11.75,4.47,False
3,4,"12,266.80",0.83,3.61,1.49,False
4,5,"8,101.70",1.32,8.40,1.41,True
5,6,"8,192.65",1.11,5.82,1.49,False
6,7,"14,791.95",0.76,9.80,-1.08,False
7,8,"10,219.03",0.93,8.88,1.96,False
8,9,"8,940.51",1.08,3.62,3.87,False
9,10,"9,215.28",1.07,2.92,3.89,False


# 6. Final quality checks

In [15]:
reporting_tables = {
    "dim_city.csv": dim_city,
    "dim_year.csv": dim_year,
    "dim_nbp_period.csv": dim_nbp_period,
    "fact_price_period.csv": fact_price_period,
    "fact_city_year.csv": fact_city_year,
    "fact_hedonic_period.csv": fact_hedonic_period,
    "city_snapshot.csv": city_snapshot,
}

expected_reporting_rows = {
    "dim_city.csv": 17,
    "dim_year.csv": 22,
    "dim_nbp_period.csv": 80,
    "fact_price_period.csv": 17 * 80,
    "fact_city_year.csv": 17 * 22,
    "fact_hedonic_period.csv": 15 * 80,
    "city_snapshot.csv": 17,
}

qa = pd.DataFrame({
    "rows": {filename: len(df) for filename, df in reporting_tables.items()},
    "expected_rows": expected_reporting_rows,
})
qa["rows_ok"] = qa["rows"] == qa["expected_rows"]

display(qa)
assert qa["rows_ok"].all()

assert not fact_price_period.duplicated(["city_id", "nbp_period_id"]).any()
assert not fact_city_year.duplicated(["city_id", "year"]).any()
assert not fact_hedonic_period.duplicated(["city_id", "nbp_period_id"]).any()
assert city_snapshot["city_id"].is_unique

city_ids = set(dim_city["city_id"])
period_ids = set(dim_nbp_period["nbp_period_id"])
years = set(dim_year["year"])

assert set(fact_price_period["city_id"]).issubset(city_ids)
assert set(fact_city_year["city_id"]).issubset(city_ids)
assert set(fact_hedonic_period["city_id"]).issubset(city_ids)
assert set(city_snapshot["city_id"]).issubset(city_ids)

assert set(fact_price_period["nbp_period_id"]).issubset(period_ids)
assert set(fact_hedonic_period["nbp_period_id"]).issubset(period_ids)
assert set(fact_city_year["year"]).issubset(years)

assert dim_city["teryt_code"].str.fullmatch(r"\d{7}").all()

assert fact_price_period[
    ["primary_transaction_price_pln_m2", "secondary_transaction_price_pln_m2"]
].min().min() > 0

# Known edge cases must survive reporting.
gdynia_id = int(dim_city.loc[dim_city["city_name"].eq("Gdynia"), "city_id"].iloc[0])
opole_id = int(dim_city.loc[dim_city["city_name"].eq("Opole"), "city_id"].iloc[0])

assert fact_city_year.loc[
    fact_city_year["year"].eq(2026),
    ["started", "completed"],
].isna().all().all(), "2026 must not appear as a full year of supply."

assert fact_city_year.loc[
    fact_city_year["city_id"].eq(gdynia_id)
    & fact_city_year["year"].eq(2005),
    "started",
].isna().all(), "Gdynia 2005 dwellings started must remain NA."

assert fact_city_year.loc[
    fact_city_year["city_id"].eq(opole_id)
    & fact_city_year["year"].eq(2007),
    "primary_annual_transaction_price_pln_m2",
].isna().all(), "Opole 2007 primary annual price must remain NA."

print("Final QA passed.")

,rows,expected_rows,rows_ok
dim_city.csv,17,17,True
dim_year.csv,22,22,True
dim_nbp_period.csv,80,80,True
fact_price_period.csv,1360,1360,True
fact_city_year.csv,374,374,True
fact_hedonic_period.csv,1200,1200,True
city_snapshot.csv,17,17,True


Final QA passed.


# 7. Export to `data/reporting/`

CSV files are saved using `utf-8-sig`.

The notebook overwrites only its own seven outputs and does not clear the entire `data/reporting/` directory.

In [16]:
for filename, df in reporting_tables.items():
    df.to_csv(
        REPORTING_DIR / filename,
        index=False,
        encoding="utf-8-sig",
    )

export_manifest = pd.DataFrame({
    "file": list(reporting_tables.keys()),
    "rows": [len(df) for df in reporting_tables.values()],
    "columns": [df.shape[1] for df in reporting_tables.values()],
    "path": [
        f"data/reporting/{filename}"
        for filename in reporting_tables.keys()
    ],
})

display(export_manifest)

,file,rows,columns,path
0,dim_city.csv,17,4,data/reporting/dim_city.csv
1,dim_year.csv,22,1,data/reporting/dim_year.csv
2,dim_nbp_period.csv,80,4,data/reporting/dim_nbp_period.csv
3,fact_price_period.csv,1360,13,data/reporting/fact_price_period.csv
4,fact_city_year.csv,374,40,data/reporting/fact_city_year.csv
5,fact_hedonic_period.csv,1200,10,data/reporting/fact_hedonic_period.csv
6,city_snapshot.csv,17,54,data/reporting/city_snapshot.csv


## 7.1 Read back quality check

In [17]:
READ_DTYPES = {
    "dim_city.csv": {"teryt_code": "string"},
}

reloaded = {
    filename: pd.read_csv(
        REPORTING_DIR / filename,
        dtype=READ_DTYPES.get(filename),
    )
    for filename in reporting_tables.keys()
}

disk_qa = pd.DataFrame({
    "rows_on_disk": {
        filename: len(df)
        for filename, df in reloaded.items()
    },
    "expected_rows": expected_reporting_rows,
})
disk_qa["ok"] = disk_qa["rows_on_disk"] == disk_qa["expected_rows"]

display(disk_qa)
assert disk_qa["ok"].all()

# Validate grain after the actual write and read cycle.
assert not reloaded["fact_price_period.csv"].duplicated(
    ["city_id", "nbp_period_id"]
).any()
assert not reloaded["fact_city_year.csv"].duplicated(
    ["city_id", "year"]
).any()
assert not reloaded["fact_hedonic_period.csv"].duplicated(
    ["city_id", "nbp_period_id"]
).any()
assert reloaded["city_snapshot.csv"]["city_id"].is_unique

# TERYT identifiers are strings and must preserve all 7 digits.
dim_city_disk = reloaded["dim_city.csv"]
assert dim_city_disk["teryt_code"].str.fullmatch(r"\d{7}").all()
assert set(dim_city_disk["teryt_code"]) == set(dim_city["teryt_code"])

# Full round trip: saved CSV files must reproduce the in memory DataFrames.
for filename, expected_df in reporting_tables.items():
    disk_df = reloaded[filename]

    pd.testing.assert_frame_equal(
        disk_df,
        expected_df.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )

print("Reporting exports saved and validated again after the full round trip.")

,rows_on_disk,expected_rows,ok
dim_city.csv,17,17,True
dim_year.csv,22,22,True
dim_nbp_period.csv,80,80,True
fact_price_period.csv,1360,1360,True
fact_city_year.csv,374,374,True
fact_hedonic_period.csv,1200,1200,True
city_snapshot.csv,17,17,True


Reporting exports saved and validated again after the full round trip.


# Power BI model

```text
dim_city
   │
   ├───────────────< fact_price_period >────────────── dim_nbp_period
   │
   ├───────────────< fact_hedonic_period >──────────── dim_nbp_period
   │
   ├───────────────< fact_city_year >───────────────── dim_year
   │
   └──────────────── city_snapshot
```

## Relationships

* `dim_city[city_id]` 1 → * `fact_price_period[city_id]`

* `dim_city[city_id]` 1 → * `fact_city_year[city_id]`

* `dim_city[city_id]` 1 → * `fact_hedonic_period[city_id]`

* `dim_city[city_id]` 1 ↔ 1 `city_snapshot[city_id]`

* `dim_nbp_period[nbp_period_id]` 1 → * `fact_price_period[nbp_period_id]`

* `dim_nbp_period[nbp_period_id]` 1 → * `fact_hedonic_period[nbp_period_id]`

* `dim_year[year]` 1 → * `fact_city_year[year]`

Cross filter direction is **Single** from dimension to fact for all fact tables.

The `dim_city` ↔ `city_snapshot` relationship uses **Both** cross filter direction.

| Question | Main table |
|---|---|
| 1. How much does an apartment cost? | `fact_price_period` |
| 2. Are wages keeping pace with prices? | `fact_city_year` |
| 3. Primary or secondary market from a price perspective? | `fact_price_period` |
| 4. Is price growth slowing or accelerating? | `fact_price_period` |
| 5. Is a lot of housing being built? | `fact_city_year` |
| 6. Which cities look relatively favorable? | `city_snapshot` |
| 7. Does the average price tell the full story? | `fact_hedonic_period` |